# Заняття 10 · LIVE DEMO 3
## RTP і SRT на тому самому поганому лінку

Емулятор лінку в обидва боки: затримка 40 мс з джиттером 10 мс (без перестановок, як на радіо), 2 % незалежних втрат, ємність 4 Mbps. RTT ≈ 80 мс — як у сценарії B (слайд 8).

| Крок | Що робимо |
|---|---|
| §1 | Лінк: емулятор у дусі `tc netem` (на Linux можна замінити справжнім netem) |
| §2 | RTP/UDP з jitter buffer 20 і 80 мс: спершу без втрат, потім з 2 % втрат |
| §3 | SRT з latency = 3 × RTT = 240 мс у MPEG-TS |
| §4 | Підсумкова таблиця «конфігурація → p50 / p99 / частка чистих кадрів» |

WebRTC у цьому ноутбуці не запускається — потрібні MediaMTX і браузер; необов’язкова клітинка наприкінці показує, як це зробити на своїй машині.

Мітка на слайдах: «LIVE DEMO 3 · lesson10_demo3_transport_netem.ipynb» (слайд 34, розділ 04 «Транспорт і UAV-лінки»); сценарій B — слайд 8, типове значення `rtpjitterbuffer latency` — слайд 12, протоколи простими словами — слайд 31, зведена таблиця протоколів — слайд 32, SRT — слайд 33. GStreamer живе в `streamlab.py` / `gstworker.py`, стиль графіків — у `panels.py`. Модель jitter buffer, FEC, NACK і SRT з повзунками — веб-демо 04 `lesson10_web04_transport.html`.

In [ ]:
# streamlab запускає GStreamer в окремому процесі (gstworker.py), тож у ноутбуці gi не імпортується
import numpy as np
import matplotlib.pyplot as plt
import streamlab as sl
from panels import PAL, set_style

set_style()
print(sl.gst_version(), '| appsink leaky-type (1.28+):', sl.appsink_has_leaky_type())

sc = sl.Scene(640, 360, 30)
ENC = 'tune=zerolatency speed-preset=superfast bitrate=1500 key-int-max=30'
SECONDS = 8

### 1. Лінк
`forward` — напрямок відео, `reverse` — зворотний канал (ACK і NAK у SRT). Обидва з тією самою затримкою і втратами.

In [ ]:
BAD = lambda: sl.Impair(rate_mbps=4, delay_ms=40, jitter_ms=10, loss=0.02, keep_order=True)
NOLOSS = lambda: sl.Impair(rate_mbps=4, delay_ms=40, jitter_ms=10, keep_order=True)

### 2–3. RTP з jitter buffer і SRT
Для кожної конфігурації — той самий потік 8 с; статистика без першої секунди розгону. «Чистий» кадр: номер читається і частка зіпсованих блоків < 2 %.

Розмір jitter buffer беремо з моделі джиттера (Gamma з k = 2, як у веб-демо 04): для джиттера σ = 10 мс частка пізніх 10⁻³ потребує ≈ 6.5 σ ≈ 65 мс, тож 80 мс — з запасом, а 20 мс — свідомо замало.

In [ ]:
CONFIGS = {
    'RTP + JB 20 · без втрат':   dict(transport='rtp', jb_ms=20, link=NOLOSS),
    'RTP + JB 80 · без втрат':   dict(transport='rtp', jb_ms=80, link=NOLOSS),
    'RTP + JB 80 · 2 % втрат':   dict(transport='rtp', jb_ms=80, link=BAD),
    'SRT 240 мс · 2 % втрат':    dict(transport='srt', srt_latency_ms=240, link=BAD),
}
if not sl.has_srt():                                   # conda-forge збирає gst-plugins-bad без libsrt
    CONFIGS = {k: v for k, v in CONFIGS.items() if v['transport'] != 'srt'}
    print('srtsrc / srtsink немає в цьому GStreamer — SRT пропускаємо (див. README; модель SRT — веб-демо 04)')
table = []
for name, cfg in CONFIGS.items():
    cfg = dict(cfg); link_f = cfg.pop('link')
    rec, tx, link = sl.run_stream(sc, SECONDS, ENC, impair=link_f(), reverse=link_f(), with_psnr=True, settle_s=1.0, **cfg)
    ok = [r for r in rec if r['idx'] in tx.t_send and r['idx'] >= 30]     # без першої секунди
    clean = [r for r in ok if r['bad'] < 0.02]
    n = len(tx.t_send) - 30
    table.append(dict(name=name, p50=sl.pct([r['age'] for r in ok], 50), p99=sl.pct([r['age'] for r in ok], 99),
                      clean=len(clean) / n, shown=len(ok) / n, lost=link.stats['forward']['lost']))
    print(f'{name:28s} готово: втрат у лінку {link.stats["forward"]["lost"]}')

### 4. Підсумок: конфігурація → p99

In [ ]:
print(f'{"конфігурація":28s} {"p50, мс":>8s} {"p99, мс":>8s} {"показано":>9s} {"чистих":>7s}')
for r in table:
    print(f'{r["name"]:28s} {r["p50"]:8.0f} {r["p99"]:8.0f} {r["shown"]:9.0%} {r["clean"]:7.0%}')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
names = [r['name'].replace(' · ', '\n') for r in table]
cols = [PAL['or_'], PAL['bl'], PAL['re'], PAL['te']][:len(table)]
ax[0].barh(names, [r['p99'] for r in table], color=cols); ax[0].set_xlabel('мс'); ax[0].set_title('Вік кадру p99')
ax[1].barh(names, [100 * r['clean'] for r in table], color=cols); ax[1].set_xlabel('%'); ax[1].set_title('Чисті кадри')
for a in ax:
    a.invert_yaxis()
ax[1].set_yticklabels([])
plt.tight_layout(); plt.show()

### Висновки для слайда
* Без втрат RTP дає найменшу затримку — але лише з jitter buffer, порахованим від виміряного джиттера: 20 мс при σ = 10 мс губить пізні пакети, і частина кадрів псується, хоча лінк нічого не втрачав. 80 мс прибирає це майже без зростання медіани: jitter buffer чекає лише тоді, коли є дірка.
* На 2 % втрат RTP без механізму відновлення непридатний для картинки: кожна втрата псує кадр і все, що на нього посилається, до наступного IDR. Більший jitter buffer рятує від пізніх пакетів, а не від втрачених.
* SRT повертає майже всі кадри чистими, але затримка стає ≥ latency (3 × RTT) + мережа: добре для backhaul у штаб, погано для керування.
* WebRTC платить за втрати третім способом — NACK/RTX на RTT і зниженням бітрейту через GCC; механізм — у таблиці протоколів (слайд 32) і у веб-демо 04.

### Необов’язково: WebRTC через MediaMTX на своїй машині
Запустіть MediaMTX (github.com/bluenviron/mediamtx), надішліть у нього RTSP, відкрийте `http://localhost:8889/cam` у браузері — це WHEP-сторінка MediaMTX. Для поганого лінку на Linux: `sudo tc qdisc add dev lo root netem delay 40ms 10ms loss 2% rate 4mbit`, після досліду — `sudo tc qdisc del dev lo root`.

In [ ]:
RUN_WEBRTC = False          # True — лише на своїй машині з MediaMTX, що слухає RTSP на 8554
if RUN_WEBRTC:
    import subprocess
    cmd = ('gst-launch-1.0 videotestsrc is-live=true ! video/x-raw,width=640,height=360,framerate=30/1 ! '
           'x264enc tune=zerolatency bitrate=1500 key-int-max=30 ! rtspclientsink location=rtsp://127.0.0.1:8554/cam')
    proc = subprocess.Popen(cmd, shell=True)
    print('відкрийте http://localhost:8889/cam у браузері; зупинка: proc.terminate()')